# SOLID Principles in Python

### 🎯 Objective
Master the **SOLID** design principles applied to idiomatic Python:
1. **S**ingle Responsibility Principle (SRP)
2. **O**pen/Closed Principle (OCP)
3. **L**iskov Substitution Principle (LSP)
4. **I**nterface Segregation Principle (ISP)
5. **D**ependency Inversion Principle (DIP)

Applying SOLID in Python leads to modular, easily testable, and loosely coupled code using standard library tools such as `abc.ABC`, `abstractmethod`, and dependency injection.


### 📖 1. The 5 SOLID Principles Explained

#### 1. Single Responsibility Principle (SRP)
*A class should have one, and only one, reason to change.* Separate domain data, business logic, and persistence.


In [ ]:
# SRP: Separate user domain logic from persistence
class User:
    def __init__(self, username: str, email: str):
        self.username = username
        self.email = email


class UserRepository:
    def __init__(self):
        self._database: dict[str, str] = {}

    def save(self, user: User) -> None:
        self._database[user.username] = user.email
        print(f"Saved user {user.username} to database.")


repo = UserRepository()
repo.save(User("alex", "alex@example.com"))


#### 2. Open/Closed Principle (OCP)
*Software entities should be open for extension, but closed for modification.* Add new behavior by implementing new classes, not by modifying existing functions with endless `if/elif`.


In [ ]:
from abc import ABC, abstractmethod


class DiscountStrategy(ABC):
    @abstractmethod
    def calculate(self, amount: float) -> float:
        pass


class RegularDiscount(DiscountStrategy):
    def calculate(self, amount: float) -> float:
        return amount


class VipDiscount(DiscountStrategy):
    def calculate(self, amount: float) -> float:
        return amount * 0.85  # 15% discount


# Adding a holiday discount requires NO modification to existing strategies
class HolidayDiscount(DiscountStrategy):
    def calculate(self, amount: float) -> float:
        return amount * 0.70  # 30% discount


#### 3. Liskov Substitution Principle (LSP)
*Subclasses should be substitutable for their base classes without altering program correctness.* Derived classes must honor the contract established by their superclass.


In [ ]:
class Bird:
    def eat(self) -> str:
        return "Eating seeds"


class FlyingBird(Bird):
    def fly(self) -> str:
        return "Flying high"


class Sparrow(FlyingBird):
    pass


class Penguin(Bird):
    # Penguin does NOT inherit from FlyingBird, avoiding invalid fly() operations!
    def swim(self) -> str:
        return "Swimming fast"


#### 4. Interface Segregation Principle (ISP)
*Clients should not be forced to depend on methods they do not use.* In Python, create focused, granular abstract interfaces or protocols.


In [ ]:
class Printable(ABC):
    @abstractmethod
    def print_document(self) -> str:
        pass


class Scannable(ABC):
    @abstractmethod
    def scan_document(self) -> str:
        pass


# Simple printer only implements Printable
class BasicPrinter(Printable):
    def print_document(self) -> str:
        return "Printing text page..."


# All-in-one printer implements both
class MultiFunctionPrinter(Printable, Scannable):
    def print_document(self) -> str:
        return "High quality print..."

    def scan_document(self) -> str:
        return "Scanning to PDF..."


#### 5. Dependency Inversion Principle (DIP)
*High-level modules should not depend on low-level modules; both should depend on abstractions.* Inject dependencies rather than hardcoding concrete classes inside constructors.


In [ ]:
class MessageSender(ABC):
    @abstractmethod
    def send(self, recipient: str, message: str) -> str:
        pass


class EmailSender(MessageSender):
    def send(self, recipient: str, message: str) -> str:
        return f"Email sent to {recipient}: {message}"


class SmsSender(MessageSender):
    def send(self, recipient: str, message: str) -> str:
        return f"SMS sent to {recipient}: {message}"


class NotificationService:
    # Depends on abstraction (MessageSender), not concrete EmailSender
    def __init__(self, sender: MessageSender):
        self._sender = sender

    def notify(self, user: str, msg: str) -> str:
        return self._sender.send(user, msg)


email_service = NotificationService(EmailSender())
print(email_service.notify("user@corp.com", "System alert"))


### 💻 2. Practice Challenge (Your Turn)

#### Challenge: Payment Processing System with OCP and DIP
Design a modular payment processing system adhering to **OCP** (open for extension) and **DIP** (inversion of control).

**Requirements**:
1. Define an abstract base class `PaymentProcessor(ABC)` with an abstract method:
   `process_payment(self, amount: float) -> tuple[bool, str]`
2. Implement two concrete processors:
   - `CreditCardProcessor`: returns `(True, f"Charged ${amount:.2f} via Credit Card")` if `amount > 0`, else `(False, "Invalid amount")`.
   - `CryptoProcessor`: has a network fee of `$5.00`. If `amount > 0`, returns `(True, f"Paid ${amount + 5.0:.2f} (includes $5 network fee) via Crypto")`, else `(False, "Invalid amount")`.
3. Implement `CheckoutService`:
   - `__init__(self, processor: PaymentProcessor) -> None` (dependency injection).
   - `checkout(self, total: float) -> str`:
     - Calls `self.processor.process_payment(total)`.
     - If success is `True`, returns `f"Order completed: {msg}"`.
     - If success is `False`, returns `f"Payment failed: {msg}"`.


In [ ]:
from abc import ABC, abstractmethod


class PaymentProcessor(ABC):
    """Abstract base interface for payment gateways."""
    @abstractmethod
    def process_payment(self, amount: float) -> tuple[bool, str]:
        pass


class CreditCardProcessor(PaymentProcessor):
    # YOUR CODE HERE
    pass


class CryptoProcessor(PaymentProcessor):
    # YOUR CODE HERE
    pass


class CheckoutService:
    # YOUR CODE HERE
    pass


# Automated Assertions
cc = CreditCardProcessor()
crypto = CryptoProcessor()

# Validate ABC inheritance
assert isinstance(cc, PaymentProcessor)
assert isinstance(crypto, PaymentProcessor)

# Validate processor responses
assert cc.process_payment(100.0) == (True, "Charged $100.00 via Credit Card")
assert cc.process_payment(-5.0) == (False, "Invalid amount")

assert crypto.process_payment(50.0) == (True, "Paid $55.00 (includes $5 network fee) via Crypto")
assert crypto.process_payment(0.0) == (False, "Invalid amount")

# Validate CheckoutService with Dependency Injection
cc_service = CheckoutService(cc)
crypto_service = CheckoutService(crypto)

assert cc_service.checkout(100.0) == "Order completed: Charged $100.00 via Credit Card"
assert cc_service.checkout(-10.0) == "Payment failed: Invalid amount"
assert crypto_service.checkout(50.0) == "Order completed: Paid $55.00 (includes $5 network fee) via Crypto"

print("✅ Challenge passed all test cases!")


<details>
<summary>💡 Hint</summary>

Inherit from `PaymentProcessor` in both `CreditCardProcessor` and `CryptoProcessor`. In `CheckoutService.__init__`, store the injected processor in `self.processor`. In `checkout()`, unpack the `(success, msg)` tuple from `self.processor.process_payment(total)`.
</details>

<details>
<summary>🔍 Solution</summary>

```python
from abc import ABC, abstractmethod


class PaymentProcessor(ABC):
    @abstractmethod
    def process_payment(self, amount: float) -> tuple[bool, str]:
        pass


class CreditCardProcessor(PaymentProcessor):
    def process_payment(self, amount: float) -> tuple[bool, str]:
        if amount <= 0:
            return (False, "Invalid amount")
        return (True, f"Charged ${amount:.2f} via Credit Card")


class CryptoProcessor(PaymentProcessor):
    def process_payment(self, amount: float) -> tuple[bool, str]:
        if amount <= 0:
            return (False, "Invalid amount")
        total_with_fee = amount + 5.0
        return (True, f"Paid ${total_with_fee:.2f} (includes $5 network fee) via Crypto")


class CheckoutService:
    def __init__(self, processor: PaymentProcessor) -> None:
        self.processor = processor

    def checkout(self, total: float) -> str:
        success, msg = self.processor.process_payment(total)
        if success:
            return f"Order completed: {msg}"
        return f"Payment failed: {msg}"
```
</details>
